In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import rand_score
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
import time
import scipy.io

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
def statistical_leverage(data):
    n, d = data.shape
    u, s, v = np.linalg.svd(data)
    l = []

    for i in range(n):
        l.append(np.linalg.norm(u[i]))
    
    return l / np.sum(l)

In [3]:
def kmeans_cost(data, centers, labels):
    cost = 0.0
    for i in range(len(data)):
        distance = np.linalg.norm(data[i] - centers[labels[i]]) ** 2
        cost += distance
    return cost

In [4]:
def get_results_kmeans(coreset_size, n_clusters, traindata, data_name):
    results = []
    kmeans = KMeans(n_clusters=n_clusters, init="k-means++").fit(traindata)
    optimal_labels = kmeans.predict(traindata)
    cost = kmeans_cost(traindata, kmeans.cluster_centers_, optimal_labels)
    probabilities = statistical_leverage(traindata)
    for ssize in coreset_size:
        tic = time.time()
        avg_cost = 0.0
        rand_index = 0.0
        for _ in range(5):
            indices = np.random.choice(traindata.shape[0], size=ssize, replace=False, p=probabilities)
            X_samples = traindata[indices]
            kmeans = KMeans(n_clusters=n_clusters, init="k-means++").fit(X_samples)
            labels = kmeans.predict(traindata)
            centers = kmeans.cluster_centers_
            avg_cost += kmeans_cost(traindata, centers, labels)
            rand_index += rand_score(optimal_labels, labels)
        rand_index /= 5
        avg_cost /= 5
        toc = time.time()
        reduction = ((traindata.shape[0] - X_samples.shape[0])/traindata.shape[0])*100
        error = (abs(avg_cost - cost)/cost)*100
        results.append({'Sampling Type': 'Statistical Leverage Sampling',
                            'Coreset Size': X_samples.shape[0],
                            'Average Cost': avg_cost,
                            'Reduction in Data Size': reduction,
                            'Error': error,
                            'Avg Rand Index': rand_index,
                            'Data': data_name,
                            'Optimal Cost': cost,
                            'Avg Time': (toc - tic)/5,
                            'Clustering Algorithm': 'KMeans++'})
    return results

In [6]:
def kmedoids_cost(data, centroids, labels):
    dist = cdist(data, centroids, 'cityblock')
    total_cost = np.sum(dist[np.arange(len(data)), labels]) 
    return total_cost

def kmediod(data, weights, k, max_iterations=500):
    data = np.asarray(data)
    
    mins = data.min(axis=0)
    maxs = data.max(axis=0)
    centroids = np.random.rand(k, data.shape[1]) * (maxs - mins) + mins
    
    for _ in range(max_iterations):
        dist = cdist(data, centroids, 'cityblock')
        weighted_dist = dist * weights[:, np.newaxis]
        labels = np.argmin(weighted_dist, axis=1)
        
        for j in range(k):
            cluster = labels == j
            if weights[cluster].sum() > 0:
                centroids[j] = np.average(data[cluster], axis=0, weights=weights[cluster])
            else:
                centroids[j] = np.random.rand(1, data.shape[1]) * (maxs - mins) + mins
    
    return centroids

def predict(data, centroids):
    dist = cdist(data, centroids, 'cityblock')
    labels = np.argmin(dist, axis=1)
    return labels

In [7]:
def get_results_kmediod(coreset_size, n_clusters, traindata, data_name):
    results = []
    centers = kmediod(traindata, np.ones(traindata.shape[0]), n_clusters)
    optimal_labels = predict(traindata, centers)
    cost = kmedoids_cost(traindata, centers, optimal_labels)
    probabilities = statistical_leverage(traindata)
    for ssize in coreset_size:
        tic = time.time()
        avg_cost = 0.0
        rand_index = 0.0
        for _ in range(5):
            indices = np.random.choice(traindata.shape[0], size=ssize, replace=False, p=probabilities)
            X_samples = traindata[indices]
            centers = kmediod(X_samples, np.ones(X_samples.shape[0]), n_clusters)
            labels = predict(traindata, centers)
            avg_cost += kmedoids_cost(traindata, centers, labels)
            rand_index += rand_score(optimal_labels, labels)
        rand_index /= 5
        avg_cost /= 5
        toc = time.time()
        reduction = ((traindata.shape[0] - X_samples.shape[0])/traindata.shape[0])*100
        error = (abs(avg_cost - cost)/cost)*100
        results.append({'Sampling Type': 'Statistical Leverage Sampling',
                            'Coreset Size': X_samples.shape[0],
                            'Average Cost': avg_cost,
                            'Reduction in Data Size': reduction,
                            'Error': error,
                            'Avg Rand Index': rand_index,
                            'Data': data_name,
                            'Optimal Cost': cost,
                            'Avg Time': (toc - tic)/5,
                            'Clustering Algorithm': 'KMediod'})
    return results

In [16]:
from sklearn.datasets import make_blobs

X, y = make_blobs(n_samples=10000, centers=50, n_features=100, random_state=42)

In [12]:
from scipy.spatial.distance import cdist

In [17]:
results = get_results_kmediod([100, 500, 1000, 1500, 3500, 5000, 7000, 10000], 50, X, 'Synthetic')

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [18]:
results

[{'Sampling Type': 'Statistical Leverage Sampling',
  'Coreset Size': 100,
  'Average Cost': 2741424.2681178115,
  'Reduction in Data Size': 99.0,
  'Error': 19.661183192562,
  'Avg Rand Index': 0.9428071447144715,
  'Data': 'Synthetic',
  'Optimal Cost': 2290988.769270514,
  'Avg Time': 1.6822498321533204,
  'Clustering Algorithm': 'KMediod'},
 {'Sampling Type': 'Statistical Leverage Sampling',
  'Coreset Size': 500,
  'Average Cost': 2495658.25305301,
  'Reduction in Data Size': 95.0,
  'Error': 8.933674687880105,
  'Avg Rand Index': 0.9572603780378037,
  'Data': 'Synthetic',
  'Optimal Cost': 2290988.769270514,
  'Avg Time': 3.6526701927185057,
  'Clustering Algorithm': 'KMediod'},
 {'Sampling Type': 'Statistical Leverage Sampling',
  'Coreset Size': 1000,
  'Average Cost': 2372862.861752891,
  'Reduction in Data Size': 90.0,
  'Error': 3.5737448206019278,
  'Avg Rand Index': 0.9639610521052104,
  'Data': 'Synthetic',
  'Optimal Cost': 2290988.769270514,
  'Avg Time': 5.379212379455

In [22]:
df = pd.read_csv('results.csv')
df = pd.concat([df, pd.DataFrame(results)], ignore_index=True)
df.to_csv('results.csv', index=False)

In [20]:
mat_data = scipy.io.loadmat('olivettifaces.mat')
traindata = mat_data['faces'].T
traindata = pd.DataFrame(traindata)
traindata.dropna()
traindata.drop_duplicates()
traindata.shape

(400, 4096)

In [21]:
results = get_results_kmediod([10, 20, 50, 70, 100], 10, traindata.values, 'Face')

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
